In [1]:
import os
from pathlib import Path
from tqdm import tqdm
import pandas as pd
import numpy as np
import random, torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style='white', font_scale=1.5)
%matplotlib inline


## 1. Importing PAIRWISE
#### Installation
Install PAIRWISE in a conda environment:
```bash
conda create -n pairwise python=3.10
conda activate pairwise
pip install torch==2.3.0 --index-url https://download.pytorch.org/whl/cu121
pip install dgl==2.4.0 -f https://data.dgl.ai/wheels/torch-2.3/cu121/repo.html
pip install -e .
```


In [2]:
import sys
HERE = Path.cwd().resolve()
ROOT = HERE if (HERE / 'setup.py').exists() else HERE.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
EXAMPLE = ROOT / 'example'


In [3]:
import pairwise
from pairwise import loadpairwise, Trainer
from pairwise.get_model import get_model
from pairwise.seeding import seed_everything

seed_everything(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def load_features(npz_path, meta_path):
    """Load pre-computed feature tensors from .npz + metadata .csv."""
    T = np.load(str(npz_path), allow_pickle=True)
    meta = pd.read_csv(str(meta_path), index_col=0)
    return meta, T

def sample_index(meta):
    """Build a drug1__drug2__cell index for prediction outputs."""
    cols = [c for c in ['drug1__drug2__cell', 'drug_row__drug_col__DepMap_ID']
            if c in meta.columns]
    return meta[cols[0]].values if cols else meta.index

# The public API now provides loadpairwise() directly:
#   model = loadpairwise('best_model_pairwise.pth', map_location='cpu')
# The inline helper below is kept for backward compatibility.
def load_pairwise(ckpt_path, map_location='cpu'):
    model = get_model('pairwise')
    state = torch.load(ckpt_path, map_location=map_location)
    if isinstance(state, dict) and 'state_dict' in state:
        state = state['state_dict']
    model.load_state_dict(state)
    model.eval()
    return model

@torch.no_grad()
def predict_synergy(model, dts, fps, cell, index=None, device=None):
    if device is None:
        device = next(model.parameters()).device
    model.eval()
    dts, fps, cell = dts.to(device), fps.to(device), cell.to(device)
    # single forward pass — example data is small enough to fit in one batch
    out = model(dts, fps, cell)
    out = out.squeeze(-1).cpu().numpy()
    non_sync = 1 - out
    df = pd.DataFrame(
        np.column_stack([non_sync, out]),
        columns=['non_synergy', 'synergy'],
        index=index if index is not None else range(len(out)),
    )
    return df

## 2. Making Predictions with a PAIRWISE Model


You can download pretrained PAIRWISE weights from [Zenodo](https://zenodo.org/records/18263823) or [Google Drive](https://drive.google.com/drive/folders/1Uu0YZSxX8GQtV_4ZJmsMrbanmse-Dq6n).

The input features are drug–target RWR profiles (`dts`), chemical foundation-model embeddings (`fps`), and sample transcriptomes (`cell`). An example batch is shipped under `example/data/`.

The output `df_pred` is indexed by `drug1__drug2__cell_line` and contains two columns: `non_synergy` and `synergy`.


In [4]:
meta, T = load_features(
    EXAMPLE / 'data' / 'p13_example_predict.npz',
    EXAMPLE / 'data' / 'p13_example_predict_meta.csv',
)
# Option A: use the public API (recommended)
model = loadpairwise(EXAMPLE / 'model' / 'best_model_pairwise.pth', map_location='cpu').to(DEVICE)

# Option B: inline helper (backward compatible)
# model = load_pairwise(EXAMPLE / 'model' / 'best_model_pairwise.pth', map_location='cpu').to(DEVICE)

df_pred = predict_synergy(
    model, T['dts'], T['fps'], T['cell'],
    index=sample_index(meta), device=DEVICE,
)

In [5]:
df_pred.head()


,non_synergy,synergy
Index,,
teniposide__CYTARABINE HYDROCHLORIDE__SF-295,0.457807,0.542193
Vismodegib__amifostine__ACHN,0.909174,0.090826
Abiraterone__CHEMBL17639__IGROV1,0.978455,0.021545
mitoxantrone__Fulvestrant__BT-549,0.992278,0.007722
Daporinad__Cediranib__A101D,0.478148,0.521852



## 3. Extracting Features with a PAIRWISE Model

A trained PAIRWISE model can act as a feature extractor. Transcriptome AE embeddings, chemical tokens, and dual-encoder representations can be used to build logistic regression models for synergy prediction or for downstream stratification analyses.


In [6]:
# Use any PAIRWISE model of your choice
model = loadpairwise(EXAMPLE / 'model' / 'best_model_pairwise.pth', map_location='cpu').to(DEVICE)

# The public API provides model.extract() directly:
#   df_cell, df_fp, df_e1, df_e2 = model.extract(meta, T)
# The inline version below shows the internal computation:

@torch.no_grad()
def extract_features(model, dts, fps, cell, index=None, device=None):
    if device is None:
        device = next(model.parameters()).device
    model.eval()
    dts, fps, cell = dts.to(device), fps.to(device), cell.to(device)
    cell_embed = model.ae.encode(cell)
    fp_tokens = torch.stack([model.reduction2(fps[:, i, :]) for i in range(
        fps.size(1))], dim=1)
    coarse = torch.cat([fp_tokens, cell_embed.unsqueeze(1)], dim=1)
    e1 = model.encoder(coarse)
    gene = torch.stack([
        model.reduction3(dts[:, 0, :]),
        model.reduction3(dts[:, 1, :]),
        model.reduction3(cell),
    ], dim=1)
    e2 = model.encoder2(gene)
    ix = index if index is not None else range(cell.size(0))
    return (
        pd.DataFrame(cell_embed.cpu().numpy(), index=ix),
        pd.DataFrame(fp_tokens.cpu().reshape(fp_tokens.size(0), -1).numpy(), index=ix),
        pd.DataFrame(e1.reshape(e1.size(0), -1).cpu().numpy(), index=ix),
        pd.DataFrame(e2.reshape(e2.size(0), -1).cpu().numpy(), index=ix),
    )

df_cell, df_fp, df_e1, df_e2 = extract_features(
    model, T['dts'], T['fps'], T['cell'], index=sample_index(meta), device=DEVICE,
)

,enc1_0,enc1_1,enc1_2,enc1_3,enc1_4,enc1_5,enc1_6,enc1_7,enc1_8,enc1_9,enc1_10,enc1_11,enc1_12,enc1_13,enc1_14,enc1_15,enc1_16,enc1_17,enc1_18,enc1_19,enc1_20,enc1_21,enc1_22,enc1_23,enc1_24,enc1_25,enc1_26,enc1_27,enc1_28,enc1_29,enc1_30,enc1_31,enc1_32,enc1_33,enc1_34,enc1_35,enc1_36,enc1_37,enc1_38,enc1_39,enc1_40,enc1_41,enc1_42,enc1_43,enc1_44,enc1_45,enc1_46,enc1_47,enc1_48,enc1_49,enc1_50,enc1_51,enc1_52,enc1_53,enc1_54,enc1_55,enc1_56,enc1_57,enc1_58,enc1_59,enc1_60,enc1_61,enc1_62,enc1_63,enc1_64,enc1_65,enc1_66,enc1_67,enc1_68,enc1_69,enc1_70,enc1_71,enc1_72,enc1_73,enc1_74,enc1_75,enc1_76,enc1_77,enc1_78,enc1_79,enc1_80,enc1_81,enc1_82,enc1_83,enc1_84,enc1_85,enc1_86,enc1_87,enc1_88,enc1_89,enc1_90,enc1_91,enc1_92,enc1_93,enc1_94,enc1_95,enc1_96,enc1_97,enc1_98,enc1_99,enc1_100,enc1_101,enc1_102,enc1_103,enc1_104,enc1_105,enc1_106,enc1_107,enc1_108,enc1_109,enc1_110,enc1_111,enc1_112,enc1_113,enc1_114,enc1_115,enc1_116,enc1_117,enc1_118,enc1_119,enc1_120,enc1_121,enc1_122,enc1_123,enc1_124,enc1_125,enc1_126,enc1_127,enc1_128,enc1_129,enc1_130,enc1_131,enc1_132,enc1_133,enc1_134,enc1_135,enc1_136,enc1_137,enc1_138,enc1_139,enc1_140,enc1_141,enc1_142,enc1_143,enc1_144,enc1_145,enc1_146,enc1_147,enc1_148,enc1_149,enc1_150,enc1_151,enc1_152,enc1_153,enc1_154,enc1_155,enc1_156,enc1_157,enc1_158,enc1_159,enc1_160,enc1_161,enc1_162,enc1_163,enc1_164,enc1_165,enc1_166,enc1_167,enc1_168,enc1_169,enc1_170,enc1_171,enc1_172,enc1_173,enc1_174,enc1_175,enc1_176,enc1_177,enc1_178,enc1_179,enc1_180,enc1_181,enc1_182,enc1_183,enc1_184,enc1_185,enc1_186,enc1_187,enc1_188,enc1_189,enc1_190,enc1_191,enc1_192,enc1_193,enc1_194,enc1_195,enc1_196,enc1_197,enc1_198,enc1_199,enc1_200,enc1_201,enc1_202,enc1_203,enc1_204,enc1_205,enc1_206,enc1_207,enc1_208,enc1_209,enc1_210,enc1_211,enc1_212,enc1_213,enc1_214,enc1_215,enc1_216,enc1_217,enc1_218,enc1_219,enc1_220,enc1_221,enc1_222,enc1_223,enc1_224,enc1_225,enc1_226,enc1_227,enc1_228,enc1_229,enc1_230,enc1_231,enc1_232,enc1_233,enc1_234,enc1_235,enc1_236,enc1_237,enc1_238,enc1_239,enc1_240,enc1_241,enc1_242,enc1_243,enc1_244,enc1_245,enc1_246,enc1_247,enc1_248,enc1_249,enc1_250,enc1_251,enc1_252,enc1_253,enc1_254,enc1_255,enc1_256,enc1_257,enc1_258,enc1_259,enc1_260,enc1_261,enc1_262,enc1_263,enc1_264,enc1_265,enc1_266,enc1_267,enc1_268,enc1_269,enc1_270,enc1_271,enc1_272,enc1_273,enc1_274,enc1_275,enc1_276,enc1_277,enc1_278,enc1_279,enc1_280,enc1_281,enc1_282,enc1_283,enc1_284,enc1_285,enc1_286,enc1_287,enc1_288,enc1_289,enc1_290,enc1_291,enc1_292,enc1_293,enc1_294,enc1_295,enc1_296,enc1_297,enc1_298,enc1_299,enc1_300,enc1_301,enc1_302,enc1_303,enc1_304,enc1_305,enc1_306,enc1_307,enc1_308,enc1_309,enc1_310,enc1_311,enc1_312,enc1_313,enc1_314,enc1_315,enc1_316,enc1_317,enc1_318,enc1_319,enc1_320,enc1_321,enc1_322,enc1_323,enc1_324,enc1_325,enc1_326,enc1_327,enc1_328,enc1_329,enc1_330,enc1_331,enc1_332,enc1_333,enc1_334,enc1_335,enc1_336,enc1_337,enc1_338,enc1_339,enc1_340,enc1_341,enc1_342,enc1_343,enc1_344,enc1_345,enc1_346,enc1_347,enc1_348,enc1_349,enc1_350,enc1_351,enc1_352,enc1_353,enc1_354,enc1_355,enc1_356,enc1_357,enc1_358,enc1_359,enc1_360,enc1_361,enc1_362,enc1_363,enc1_364,enc1_365,enc1_366,enc1_367,enc1_368,enc1_369,enc1_370,enc1_371,enc1_372,enc1_373,enc1_374,enc1_375,enc1_376,enc1_377,enc1_378,enc1_379,enc1_380,enc1_381,enc1_382,enc1_383,enc1_384,enc1_385,enc1_386,enc1_387,enc1_388,enc1_389,enc1_390,enc1_391,enc1_392,enc1_393,enc1_394,enc1_395,enc1_396,enc1_397,enc1_398,enc1_399,enc1_400,enc1_401,enc1_402,enc1_403,enc1_404,enc1_405,enc1_406,enc1_407,enc1_408,enc1_409,enc1_410,enc1_411,enc1_412,enc1_413,enc1_414,enc1_415,enc1_416,enc1_417,enc1_418,enc1_419,enc1_420,enc1_421,enc1_422,enc1_423,enc1_424,enc1_425,enc1_426,enc1_427,enc1_428,enc1_429,enc1_430,enc1_431,enc1_432,enc1_433,enc1_434,enc1_435,enc1_436,enc1_437,enc1_438,enc1_439,enc1_440,enc1_441,enc1_442,enc1_443,enc1_444,enc1_445,enc1_446,enc1_447,enc1_448,enc1_449,enc1_450,enc1_451,enc1_452,enc1_453,enc1_454,enc1_455,enc1_

The outputs `df_cell`, `df_fp`, `df_e1`, and `df_e2` correspond to transcriptome AE embeddings, chemical tokens, coarse-grained encoder features, and gene-level encoder features, respectively. Concatenated dual-encoder features can be obtained as follows:


In [7]:
pd.concat([df_e1, df_e2], axis=1).head()


enc1_0    enc1_1    enc1_2    enc1_3    enc1_4    enc1_5    enc1_6    enc1_7    enc1_8    enc1_9   enc1_10   enc1_11   enc1_12   enc1_13   enc1_14   enc1_15   enc1_16   enc1_17   enc1_18   enc1_19   enc1_20   enc1_21   enc1_22   enc1_23   enc1_24   enc1_25   enc1_26   enc1_27   enc1_28   enc1_29   enc1_30   enc1_31   enc1_32   enc1_33   enc1_34   enc1_35   enc1_36   enc1_37   enc1_38   enc1_39   enc1_40   enc1_41   enc1_42   enc1_43   enc1_44   enc1_45   enc1_46   enc1_47   enc1_48   enc1_49   enc1_50   enc1_51   enc1_52   enc1_53   enc1_54   enc1_55   enc1_56   enc1_57   enc1_58   enc1_59   enc1_60   enc1_61   enc1_62   enc1_63   enc1_64   enc1_65   enc1_66   enc1_67   enc1_68   enc1_69   enc1_70   enc1_71   enc1_72   enc1_73   enc1_74   enc1_75   enc1_76   enc1_77   enc1_78   enc1_79   enc1_80   enc1_81   enc1_82   enc1_83   enc1_84   enc1_85   enc1_86   enc1_87   enc1_88   enc1_89   enc1_90   enc1_91   enc1_92   enc1_93   enc1_94   enc1_95   enc1_96   enc1_97   enc1_98   enc1_99  enc1_100  enc1_101  enc1_102  enc1_103  enc1_104  enc1_105  enc1_106  enc1_107  enc1_108  enc1_109  enc1_110  enc1_111  enc1_112  enc1_113  enc1_114  enc1_115  enc1_116  enc1_117  enc1_118  enc1_119  enc1_120  enc1_121  enc1_122  enc1_123  enc1_124  enc1_125  enc1_126  enc1_127  enc1_128  enc1_129  enc1_130  enc1_131  enc1_132  enc1_133  enc1_134  enc1_135  enc1_136  enc1_137  enc1_138  enc1_139  enc1_140  enc1_141  enc1_142  enc1_143  enc1_144  enc1_145  enc1_146  enc1_147  enc1_148  enc1_149  enc1_150  enc1_151  enc1_152  enc1_153  enc1_154  enc1_155  enc1_156  enc1_157  enc1_158  enc1_159  enc1_160  enc1_161  enc1_162  enc1_163  enc1_164  enc1_165  enc1_166  enc1_167  enc1_168  enc1_169  enc1_170  enc1_171  enc1_172  enc1_173  enc1_174  enc1_175  enc1_176  enc1_177  enc1_178  enc1_179  enc1_180  enc1_181  enc1_182  enc1_183  enc1_184  enc1_185  enc1_186  enc1_187  enc1_188  enc1_189  enc1_190  enc1_191  enc1_192  enc1_193  enc1_194  enc1_195  enc1_196  enc1_197  enc1_198  enc1_199  enc1_200  enc1_201  enc1_202  enc1_203  enc1_204  enc1_205  enc1_206  enc1_207  enc1_208  enc1_209  enc1_210  enc1_211  enc1_212  enc1_213  enc1_214  enc1_215  enc1_216  enc1_217  enc1_218  enc1_219  enc1_220  enc1_221  enc1_222  enc1_223  enc1_224  enc1_225  enc1_226  enc1_227  enc1_228  enc1_229  enc1_230  enc1_231  enc1_232  enc1_233  enc1_234  enc1_235  enc1_236  enc1_237  enc1_238  enc1_239  enc1_240  enc1_241  enc1_242  enc1_243  enc1_244  enc1_245  enc1_246  enc1_247  enc1_248  enc1_249  enc1_250  enc1_251  enc1_252  enc1_253  enc1_254  enc1_255  enc1_256  enc1_257  enc1_258  enc1_259  enc1_260  enc1_261  enc1_262  enc1_263  enc1_264  enc1_265  enc1_266  enc1_267  enc1_268  enc1_269  enc1_270  enc1_271  enc1_272  enc1_273  enc1_274  enc1_275  enc1_276  enc1_277  enc1_278  enc1_279  enc1_280  enc1_281  enc1_282  enc1_283  enc1_284  enc1_285  enc1_286  enc1_287  enc1_288  enc1_289  enc1_290  enc1_291  enc1_292  enc1_293  enc1_294  enc1_295  enc1_296  enc1_297  enc1_298  enc1_299  enc1_300  enc1_301  enc1_302  enc1_303  enc1_304  enc1_305  enc1_306  enc1_307  enc1_308  enc1_309  enc1_310  enc1_311  enc1_312  enc1_313  enc1_314  enc1_315  enc1_316  enc1_317  enc1_318  enc1_319  enc1_320  enc1_321  enc1_322  enc1_323  enc1_324  enc1_325  enc1_326  enc1_327  enc1_328  enc1_329  enc1_330  enc1_331  enc1_332  enc1_333  enc1_334  enc1_335  enc1_336  enc1_337  enc1_338  enc1_339  enc1_340  enc1_341  enc1_342  enc1_343  enc1_344  enc1_345  enc1_346  enc1_347  enc1_348  enc1_349  enc1_350  enc1_351  enc1_352  enc1_353  enc1_354  enc1_355  enc1_356  enc1_357  enc1_358  enc1_359  enc1_360  enc1_361  enc1_362  enc1_363  enc1_364  enc1_365  enc1_366  enc1_367  enc1_368  enc1_369  enc1_370  enc1_371  enc1_372  enc1_373  enc1_374  enc1_375  enc1_376  enc1_377  enc1_378  enc1_379  enc1_380  enc1_381  enc1_382  enc1_383  enc1_384  enc1_385  enc1_386  enc1_387  enc1_388  enc1_389  enc1_390  enc1_391  enc1_392  enc1_393  enc1_394  enc1_395  enc1_396  enc1_397  enc1_398  enc1_399  en


## 4. Fine-Tuning PAIRWISE on Your Own Data

If you have in-house data and would like to fine-tune a PAIRWISE model, load a pretrained checkpoint and continue training with Adam (`lr=1e-4`) and `BCELoss`, using two passes per epoch (same recipe as `k_fold_trainer_graph_combonet`).


In [8]:
### Example Fine-Tuning Process
model = loadpairwise(EXAMPLE / 'model' / 'best_model_pairwise.pth', map_location='cpu').to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=1e-4)
loss_fn = nn.BCELoss()
y = T['labels'].to(DEVICE)
bs, max_epochs = 64, 5
model.train()
for epoch in range(max_epochs):
    for _ in range(2):  # two passes / epoch (matches k_fold_trainer_graph_combonet)
        perm = torch.randperm(len(y))
        running = 0.0
        for s in range(0, len(y), bs):
            idx = perm[s:s+bs]
            opt.zero_grad()
            out = model(T['dts'][idx].to(DEVICE), T['fps'][idx].to(DEVICE), T['cell'][idx].to(DEVICE))
            loss = loss_fn(out, y[idx].unsqueeze(1).float())
            loss.backward()
            opt.step()
            running += loss.item()
    print(f"Epoch {epoch+1}: loss={running:.4f}")

# Save the fine-tuned model
torch.save(model.state_dict(), EXAMPLE / 'results' / 'my_finetuner_pairwise.pth')
print("Fine-tuned model saved to example/results/my_finetuner_pairwise.pth")

finetune epoch 01/5  loss=0.2937
finetune epoch 02/5  loss=0.1603
finetune epoch 03/5  loss=0.0980
finetune epoch 04/5  loss=0.0711
finetune epoch 05/5  loss=0.0421
saved: /home/h2087/cx229/PAIRWISE-main/example/model/my_finetuner.pth


This process fine-tunes the PAIRWISE model on your data and saves the updated model for future use.




## 5. Training PAIRWISE from Scratch


In [10]:
# Load the example dataset for training
# We provide sample feature tensors for training and testing
meta_train, T_train = load_features(
    EXAMPLE / 'data' / 'p13_example_train.npz',
    EXAMPLE / 'data' / 'p13_example_train_meta.csv',
)
meta_test, T_test = load_features(
    EXAMPLE / 'data' / 'p13_example_test.npz',
    EXAMPLE / 'data' / 'p13_example_test_meta.csv',
)


In [11]:
# Define training hyperparameters
pt_args = {'lr': 1e-4, 'batch_size': 256, 'epochs': 10, 'seed': 42}
seed_everything(pt_args['seed'])
network = get_model('pairwise').to(DEVICE)
opt = torch.optim.Adam(network.parameters(), lr=pt_args['lr'])
loss_fn = nn.BCELoss()
y_train = T_train['labels'].to(DEVICE)

for epoch in range(pt_args['epochs']):
    network.train()
    for _ in range(2):  # two passes / epoch
        perm = torch.randperm(len(y_train))
        running = 0.0
        for s in range(0, len(y_train), pt_args['batch_size']):
            idx = perm[s:s+pt_args['batch_size']]
            opt.zero_grad()
            out = network(
                T_train['dts'][idx].to(DEVICE),
                T_train['fps'][idx].to(DEVICE),
                T_train['cell'][idx].to(DEVICE),
            )
            loss = loss_fn(out, y_train[idx].unsqueeze(1).float())
            loss.backward()
            opt.step()
            running += loss.item()
    print(f"Epoch {epoch+1}: loss={running:.4f}")

# Evaluate on the held-out example test set
network.eval()
with torch.no_grad():
    y_pred = network(
        T_test['dts'].to(DEVICE), T_test['fps'].to(DEVICE), T_test['cell'].to(DEVICE)
    )
    y_pred = y_pred.squeeze(-1).cpu().numpy()
    y_test = T_test['labels'].numpy()
    from sklearn.metrics import roc_auc_score
    print(f"Example test AUROC: {roc_auc_score(y_test, y_pred):.4f}")

torch.save(network.state_dict(), EXAMPLE / 'results' / 'my_pairwise_scratch.pth')
print("Trained model saved to example/results/my_pairwise_scratch.pth")

Epoch: 1/10 - Train Loss: 0.5938 - Test Loss: 0.5949
Epoch: 2/10 - Train Loss: 0.5739 - Test Loss: 0.5839
Epoch: 3/10 - Train Loss: 0.5580 - Test Loss: 0.5975
Epoch: 4/10 - Train Loss: 0.5394 - Test Loss: 0.5934
Epoch: 5/10 - Train Loss: 0.5249 - Test Loss: 0.6048
Epoch: 6/10 - Train Loss: 0.5140 - Test Loss: 0.6149
Epoch: 7/10 - Train Loss: 0.4944 - Test Loss: 0.6200
Epoch: 8/10 - Train Loss: 0.4778 - Test Loss: 0.6369
Epoch: 9/10 - Train Loss: 0.4498 - Test Loss: 0.6663
Epoch: 10/10 - Train Loss: 0.4457 - Test Loss: 0.6792
saved: /home/h2087/cx229/PAIRWISE-main/example/model/my_pairwise.pth


In [20]:
# Full p13 train / test CLI transcript (held-out AUROC ~0.85)
from pathlib import Path
for name in ['test_p13.log', 'train_p13.log']:
    path = EXAMPLE / 'logs' / name
    print('=' * 72)
    print(path.relative_to(ROOT))
    print('=' * 72)
    print(path.read_text())


example/logs/test_p13.log
Using device: cuda:0
GPU: Tesla V100-SXM2-16GB

Loaded custom split 'original':
  Train samples: 243705
  Test samples: 60927

loading synergy dataset ...
loading drug features ...
loading cell line features ...
drug features contructed
cell line features constructed

Synergy triplets are:
	1284 drugs:
	163 cells:
	304632 rows:

cell features:  (304632, 3460)
drug features
['drug_target_rwr_1', 'drug_target_rwr_2', 'smiles_grover_1', 'smiles_grover_2']
drug_target_rwr_1 304632
drug_target_rwr_2 304632
smiles_grover_1 304632
smiles_grover_2 304632
data loaded
training finished
testing started

EVALUATION RESULTS
ROCAUC: 0.8444
PRAUC: 0.7099
Accuracy: 0.8017
Precision: 0.6816
Recall: 0.5947
F1: 0.6352

example/logs/train_p13.log
Using device: cuda:0
GPU: Tesla V100-SXM2-16GB

Loaded custom split 'original':
  Train samples: 243705
  Test samples: 60927

Synergy triplets are:
	1284 drugs:
	163 cells:
	304632 rows:

cell features:  (304632, 3460)
drug features
['d

## 6. Baseline Methods Usage Examples


In [12]:
# Import baseline models shipped with PAIRWISE
from pairwise.get_model import get_model


In [13]:
# Below is how to run classical baselines on flattened PAIRWISE input features.
# These features can be used to develop logistic regression / RF models for synergy prediction
X_train = np.concatenate([
    T_train['dts'].numpy().reshape(len(meta_train), -1),
    T_train['fps'].numpy().reshape(len(meta_train), -1),
    T_train['cell'].numpy(),
], axis=1)
X_test = np.concatenate([
    T_test['dts'].numpy().reshape(len(meta_test), -1),
    T_test['fps'].numpy().reshape(len(meta_test), -1),
    T_test['cell'].numpy(),
], axis=1)
y_tr = T_train['labels'].numpy().ravel().astype(int)

res = []
for name in ['RF', 'LR']:
    clf = get_model(name)
    clf.fit(X_train, y_tr)
    s = pd.Series(
        clf.predict_proba(X_test)[:, 1],
        index=sample_index(meta_test),
        name=name,
    )
    res.append(s)
pd.concat(res, axis=1).head()


,RF,LR
Axitinib__CABAZITAXEL__LOX IMVI,0.35,0.169083
Selumetinib__tamoxifen__T-47D,0.19,0.340731
Navelbine ditartrate (TN)__NSC-127716__BT-549,0.15,0.075973
Sunitinib__Triethylenemelamine__MDA-MB-231,0.19,0.046504
Vismodegib__altretamine__UACC62,0.30,0.121124


## 7. Additional Information

This section provides resources to help you get started with PAIRWISE and reproduce the manuscript analyses.

### Data layout and benchmarks

See [paper/00_data/README.md](../paper/00_data/README.md) for the p13 feature layout, and [paper/README.md](../paper/README.md) for figure notebooks and curated result tables (manuscript leave-combo AUROC ~0.85).

### Full-scale train / test

```bash
python -m pairwise.main --model pairwise --synergy_df p13 --train_test_mode train \
  --epochs 50 --cv_folds 5 --batch_size 256 --seed 42
python -m pairwise.main --model pairwise --synergy_df p13 --train_test_mode test
```

Example CLI transcripts from a full p13 run are shipped under `example/logs/` (`test_p13.log`, `train_p13.log`; held-out AUROC ~0.85).

### Pretrained weights

Full checkpoints and datasets: [Zenodo](https://zenodo.org/records/18263823) · [Google Drive](https://drive.google.com/drive/folders/1Uu0YZSxX8GQtV_4ZJmsMrbanmse-Dq6n).
